In [ ]:
import pandas as pd
from datasets import load_dataset
from utils.divide_dataset import divide_dataset_train_val_test
from tqdm.auto import tqdm
from tokenizer.hugging_face_tokenizer_wrapper import HFTokenizerWrapper
from tokenizer.hugging_face_tokenizer import train_hugging_face_tokenizer

c:\Users\Manuel Fas Ruiz\Machine_Learning\Paper-Reimplementations\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_set_c4 = load_dataset("allenai/c4", "realnewslike")

In [3]:
len(data_set_c4["validation"]), len(data_set_c4["train"])

(13863, 13799838)

In [4]:
train_ds, val_ds, test_ds = divide_dataset_train_val_test(data_set_c4, test_size=0.5, seed=42)


In [5]:
len(train_ds), len(val_ds), len(test_ds)

(13799838, 6931, 6932)

In [6]:
train = data_set_c4["train"]
print(train)                     # number of documents in the full split

# About 5MB of text, taken from the start
sample_texts, total_chars = [], 0
for text in train.select(range(20_000))["text"]:
    sample_texts.append(text)
    total_chars += len(text)
    if total_chars >= 5_000_000:
        break
print(len(sample_texts), "documents,", total_chars, "characters")

Dataset({
    features: ['text', 'timestamp', 'url'],
    num_rows: 13799838
})
1862 documents, 5005692 characters


In [7]:
import json
with open("c4_sample_5mb.json", "w", encoding="utf-8") as f:
    json.dump(sample_texts, f)

In [11]:
wrapper = HFTokenizerWrapper(path="tokenizer/tokenizer.json")

In [12]:
wrapper.encode("Hello Manuel Fas")

[142, 171, 178, 178, 1, 147, 167, 180, 187, 171, 1, 140, 167, 1]

In [ ]:
from datasets import load_dataset
from tokenizer.hugging_face_tokenizer_wrapper import HFTokenizerWrapper   # adjust to where your wrapper lives

val = load_dataset("allenai/c4", "realnewslike", split="validation")
wrapper = HFTokenizerWrapper(path="tokenizer/tokenizer.json")

# 1. Size and special token IDs
print("vocab size:", wrapper.vocab_size)                                   # 8192
print("pad / unk / eos:", wrapper.pad_id, wrapper.unk_id, wrapper.eos_id)  # 0 1 2
print("sentinels:", wrapper.sentinel_ids[0], "...", wrapper.sentinel_ids[-1])  # 3 ... 102

# 2. How a sentence is split
ids = wrapper.encode("The government announced new economic measures on Tuesday.")
print([wrapper.tokinzer.id_to_token(i) for i in ids])     # .tokinzer if you kept that name

# 3. Round trip, UNK rate, tokens per word on 500 validation documents
texts = val.select(range(500))["text"]
failed = [t for t in texts if wrapper.decode(wrapper.encode(t)) != " ".join(t.split())]
print("round trip failures:", len(failed), "of", len(texts))

all_ids = [i for t in texts for i in wrapper.encode(t)]
print("UNK rate:", sum(i == wrapper.unk_id for i in all_ids) / len(all_ids))
print("tokens per word:", len(all_ids) / sum(len(t.split()) for t in texts))

# 4. Sentinels decode as separate words
test = wrapper.encode("Thank you") + [wrapper.sentinel_ids[0]] + wrapper.encode("me to your party") + [wrapper.eos_id]
print(wrapper.decode(test))      # Thank you <extra_id_0> me to your party [EOS]

c:\Users\Manuel Fas Ruiz\Machine_Learning\Paper-Reimplementations\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab size: 8192
pad / unk / eos: 0 1 2
sentinels: 3 ... 102
['The[END_WORD]', 'government[END_WORD]', 'announced[END_WORD]', 'new[END_WORD]', 'economic[END_WORD]', 'measures[END_WORD]', 'on[END_WORD]', 'Tuesday.[END_WORD]']
round trip failures: 4 of 500
UNK rate: 3.11478844923178e-05
tokens per word: 1.6124060030225136
Thank you <extra_id_0> me to your party [EOS]


In [3]:
from dataset.span_corruption import sample_span_lenghts, corrupt_spans



In [15]:
test_lenghts

[2, 6, 7]

In [17]:
# Example sentence
text = "The transformer architecture changed natural language processing and enabled large language models"
tokens = text.split()

corrupted, target = corrupt_spans(
    tokens, sentinel_ids=wrapper.sentinel_ids, eos_id=wrapper.eos_id,corruption_rate=0.15, mean_span_lenght=3, seed=42
)

original_length = len(tokens)
corrupted_length = len(corrupted)

In [18]:
original_length, corrupted_length

(12, 13)

In [19]:
corrupted

['The',
 'transformer',
 'architecture',
 'changed',
 'natural',
 'language',
 'processing',
 'and',
 'enabled',
 'large',
 3,
 'models',
 2]

In [20]:
target 

[3, 'language', 2]

In [4]:
import numpy as np

S = set(wrapper.sentinel_ids)

def reconstruct(inp, tgt):
    spans, cur = {}, None
    for t in tgt[:-1]:                      # skip EOS
        if t in S:
            cur = t
            spans[cur] = []
        else:
            spans[cur].append(t)
    out = []
    for t in inp[:-1]:                      # skip EOS
        out.extend(spans[t] if t in S else [t])
    return out

# Look at one example
ids = wrapper.encode(val[0]["text"])[:512]
inp, tgt = corrupt_spans(ids, wrapper.sentinel_ids, wrapper.eos_id, seed=0)
print("INPUT: ", wrapper.decode(inp)[:300])
print("TARGET:", wrapper.decode(tgt)[:300])

# Test on 300 real documents
ok, fracs = True, []
for i in range(300):
    ids = wrapper.encode(val[i]["text"])[:512]
    inp, tgt = corrupt_spans(ids, wrapper.sentinel_ids, wrapper.eos_id)
    ok &= reconstruct(inp, tgt) == ids
    n_spans = sum(t in S for t in tgt)
    fracs.append((len(tgt) - 1 - n_spans) / len(ids))
print("all reconstruct:", ok, f"| masked: {np.mean(fracs):.1%}")

INPUT:  Cluster comprises IBM's Opteron-based eServer 325 serv <extra_id_0> management software and storage devices that can run Linux and Windows operating systems. IBM on Tuesday announced a prepackaged and pretested cluster that is powered by Advanced Micro Devices Inc.s 64-bit Opteron process <extra_id_
TARGET: <extra_id_0> er and systems <extra_id_1> or. IBM, of <extra_id_2> —with systems management software <extra_id_3> run Lin <extra_id_4> of IB <extra_id_5> e <extra_id_6> and two-processor <extra_id_7> using Intel Corp. <extra_id_8> Xe <extra_id_9> on ch <extra_id_10> ched interconnects for super <extr
all reconstruct: True | masked: 14.8%


In [22]:
import os, time, json, tokenizers

p = "tokenizer/tokenizer.json"
print("file:", os.path.abspath(p))
print("last written:", time.ctime(os.path.getmtime(p)))
print("tokenizers version:", tokenizers.__version__)

cfg = json.load(open(p, encoding="utf-8"))
vocab = cfg["model"]["vocab"]
print("model suffix:", cfg["model"]["end_of_word_suffix"])
print("pre-tokenizer:", cfg["pre_tokenizer"])
print("vocab size:", len(vocab))
print("tokens ending in [END_WORD]:", sum(k.endswith("[END_WORD]") for k in vocab))
print("'s[END_WORD]' in vocab:", "s[END_WORD]" in vocab)

file: c:\Users\Manuel Fas Ruiz\Machine_Learning\Paper-Reimplementations\06_Switch_Transformers_MOE\tokenizer\tokenizer.json
last written: Tue Nov 24 11:28:06 2026
tokenizers version: 0.23.3
model suffix: [END_WORD]
pre-tokenizer: {'type': 'WhitespaceSplit'}
vocab size: 211
tokens ending in [END_WORD]: 5
's[END_WORD]' in vocab: False


In [106]:
import numpy as np
from tokenizer.hugging_face_tokenizer_wrapper import HFTokenizerWrapper 
data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
wrapper = HFTokenizerWrapper(path="tokenizer/tokenizer.json")
print(len(data))                                       # 4718318
print((data == wrapper.eos_id).sum())                  # 6931, one EOS per document

9448288457
13799839


In [9]:
from dataset.dataset import C4SpanCorruptionDataset



In [10]:

import numpy as np
from tokenizer.hugging_face_tokenizer_wrapper import HFTokenizerWrapper 


In [11]:
wrapper.pad_id, wrapper.eos_id, wrapper.unk_id


NameError: name 'wrapper' is not defined

In [59]:

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [13]:
type(wrapper.pad_id)

NameError: name 'wrapper' is not defined

In [14]:
from dataset.dataloaders import create_dataloaders


SyntaxError: invalid syntax. Perhaps you forgot a comma? (dataloaders.py, line 18)

In [15]:
wrapper = HFTokenizerWrapper(path="tokenizer/tokenizer.json")                 # 6931, one EOS per document
train_dataset = C4SpanCorruptionDataset("data/train.bin", wrapper.sentinel_ids, wrapper.eos_id)
val_dataset = C4SpanCorruptionDataset("data/val.bin", wrapper.sentinel_ids, wrapper.eos_id)
test_dataset = C4SpanCorruptionDataset("data/test.bin", wrapper.sentinel_ids, wrapper.eos_id)


In [60]:
from dataset.dataloaders import create_dataloaders
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, val_dataset, test_dataset, wrapper.pad_id, num_workers=1, batch_size=512)

In [61]:
batch = next(iter(train_loader))

In [68]:
batch.keys()

dict_keys(['inputs_ids', 'attention_mask', 'decoder_inputs_ids', 'labels'])

In [69]:
batch["attention_mask"]

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)

<>:1: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
<>:1: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
C:\Users\Manuel Fas Ruiz\AppData\Local\Temp\ipykernel_35236\4132625387.py:1: SyntaxWarning: list indices must be integers or slices, not tuple; perhaps you missed a comma?
  [T, T][:, 1:]


TypeError: list indices must be integers or slices, not tuple

In [19]:
import torch

from model.layers import MultiHeadAttention
from model.encoder import EncoderBlock, Encoder

mha= MultiHeadAttention(num_heads=8, head_size=64, n_embeddings=512, dropout=0.0)
enc = torch.randn(2, 10, 512)
dec = torch.randn(2, 6, 512)
enc_mask = torch.ones(2, 1, 10)

out, w = mha(enc, enc, enc_mask)
print(out.shape, w.shape)    # (2, 10, 512)  (2, 8, 10, 10)
out, w = mha(dec, enc, enc_mask)
print(out.shape, w.shape)    # (2, 6, 512)   (2, 8, 6, 10)

torch.Size([2, 10, 512]) torch.Size([2, 8, 10, 10])
torch.Size([2, 6, 512]) torch.Size([2, 8, 6, 10])


In [41]:


encoder = Encoder(20, num_embeddings=512, sequence_lenght=512, num_heads=8, n_layers=8, dimensionality_ff=512, dropout=0.0 )

In [42]:
encoder

Encoder(
  (token_embedding_table): Embedding(20, 512)
  (positional_encoding): Embedding(512, 512)
  (encoder_blocks): ModuleList(
    (0-7): 8 x EncoderBlock(
      (multi_head_attention): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x AttentionHead(
            (queries): Linear(in_features=512, out_features=64, bias=False)
            (keys): Linear(in_features=512, out_features=64, bias=False)
            (values): Linear(in_features=512, out_features=64, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (proj): Linear(in_features=512, out_features=512, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (fnn): MLP(
        (fnn): Sequential(
          (0): Linear(in_features=512, out_features=512, bias=False)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=512, bias=False)
          (3): Dropout(p=0.0, inplace=False)
        )
      )
      (ln1): RMSNorm((512,), e

In [46]:
from model.decoder import DecoderBlock

dec = DecoderBlock(number_heads=8,  n_embeddings=512, dimensionality_ff=512, dropout=0.0)
decoder = torch.randn(2, 6, 512)
encoder_output = torch.randn(2, 10, 512)
casual_mask = torch.randn(6,6)
enc_mask = torch.ones(2, 1, 10)

out, w = dec(decoder, enc_mask, casual_mask, encoder_output)

In [47]:
out.shape

torch.Size([2, 6, 512])

In [48]:
dec

DecoderBlock(
  (self_attention): MultiHeadAttention(
    (heads): ModuleList(
      (0-7): 8 x AttentionHead(
        (queries): Linear(in_features=512, out_features=64, bias=False)
        (keys): Linear(in_features=512, out_features=64, bias=False)
        (values): Linear(in_features=512, out_features=64, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (proj): Linear(in_features=512, out_features=512, bias=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (ln1): RMSNorm((512,), eps=None, elementwise_affine=True)
  (cross_attention): MultiHeadAttention(
    (heads): ModuleList(
      (0-7): 8 x AttentionHead(
        (queries): Linear(in_features=512, out_features=64, bias=False)
        (keys): Linear(in_features=512, out_features=64, bias=False)
        (values): Linear(in_features=512, out_features=64, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (proj): Linear(in_features=512, out_features=512, bias=False)
